# Simplified Research Implementation
Public, incomplete implementation. No patient data, outputs, metrics, or reproduced results are included.

This notebook preserves the LSTM architecture and MC Dropout logic while requiring explicit, authorized data access. It does not reproduce the full research pipeline.


## Data safety and methodological warnings
- `DATA_DIR = None` is a safe placeholder. Data reading functions are never called automatically.
- Split by `subject_id` rather than ICU stay to reduce patient leakage; apply a temporal cutoff before label construction.
- Fit imputation, scaling, feature selection, and every other preprocessing step on training data only.
- Item IDs and labels are retained as provisional, stay-level retrospective constructs; they are not validated future targets or clinical claims.
- Full preprocessing, sequence construction, training, calibration, evaluation, and leakage review remain incomplete.


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

DATA_DIR = None
REQUIRED_FILES = ['PATIENTS.csv', 'ADMISSIONS.csv', 'ICUSTAYS.csv', 'CHARTEVENTS.csv', 'LABEVENTS.csv', 'INPUTEVENTS_CV.csv', 'INPUTEVENTS_MV.csv', 'OUTPUTEVENTS.csv', 'PROCEDUREEVENTS_MV.csv']
CHART_ITEM_IDS = [211, 220045, 618, 220210, 646, 220277, 51, 455, 220179, 676, 223762]
LAB_ITEM_IDS = [50912, 50820, 50813, 50882, 50931, 51006, 51222, 50818, 50821, 50902]
ITEMID_MAP = {211: 'heart_rate', 220045: 'heart_rate', 618: 'resp_rate', 220210: 'resp_rate', 646: 'spo2', 220277: 'spo2', 51: 'sbp', 455: 'sbp', 220179: 'sbp', 223762: 'temperature', 676: 'temperature'}
LAB_ITEMID_MAP = {50912: 'creatinine', 50820: 'ph', 50813: 'lactate', 50882: 'bicarbonate', 50931: 'glucose', 51006: 'bun', 51222: 'hemoglobin', 50818: 'pco2', 50821: 'po2', 50902: 'chloride'}
VITAL_COLS = ['heart_rate', 'resp_rate', 'spo2', 'sbp', 'temperature']
LAB_COLS = ['creatinine', 'ph', 'lactate', 'bicarbonate', 'glucose', 'bun', 'hemoglobin', 'pco2', 'po2', 'chloride']
IO_COLS = ['fluid_intake_per_hour', 'urine_output_per_hour']
ALL_FEATURE_COLS = VITAL_COLS + LAB_COLS + IO_COLS
WINDOW_SIZE = 24
N_MC_SAMPLES = 50
print('Definitions loaded. DATA_DIR is None; no data or model loaded.')


In [ ]:
def read_csv_required(path, required_columns, **kwargs):
    unsupported = {'usecols', 'iterator', 'chunksize', 'nrows', 'header', 'names'} & kwargs.keys()
    if unsupported:
        raise ValueError(f'read_csv_required manages {sorted(unsupported)}; omit these arguments')
    header = pd.read_csv(path, nrows=0, **kwargs)
    names = {str(column).strip().lower(): column for column in header.columns}
    missing = [column for column in required_columns if column.lower() not in names]
    if missing:
        raise ValueError(f'Missing required columns: {missing}')
    frame = pd.read_csv(path, usecols=[names[column.lower()] for column in required_columns], **kwargs)
    frame.columns = [str(column).strip().lower() for column in frame.columns]
    return frame

def load_authorized_tables(data_dir, filenames=REQUIRED_FILES):
    if data_dir is None:
        raise ValueError('Pass an authorized data_dir explicitly')
    data_dir = Path(data_dir)
    return {filename: pd.read_csv(data_dir / filename, low_memory=False) for filename in filenames}

def parse_nullable_numeric(frame, columns):
    result = frame.copy()
    for column in columns:
        if column in result:
            dtype = 'Int64' if column.endswith('_id') or column == 'itemid' else 'Float64'
            result[column] = pd.to_numeric(result[column], errors='coerce').astype(dtype)
    return result

def parse_timestamps(frame, columns):
    result = frame.copy()
    for column in columns:
        if column in result:
            result[column] = pd.to_datetime(result[column], errors='coerce')
    return result

def read_filtered_events(data_dir, filename, columns, item_ids, chunksize=500000):
    if data_dir is None:
        raise ValueError('Pass an authorized data_dir explicitly')
    if not isinstance(chunksize, int) or isinstance(chunksize, bool) or chunksize <= 0:
        raise ValueError('chunksize must be a positive integer')
    data_dir = Path(data_dir)
    header = pd.read_csv(data_dir / filename, nrows=0)
    names = {str(column).strip().lower(): column for column in header.columns}
    required = [str(column).strip().lower() for column in columns]
    if 'itemid' not in required or 'valuenum' not in required:
        raise ValueError('Event columns must include itemid and valuenum')
    missing = [column for column in required if column not in names]
    if missing:
        raise ValueError(f'Missing required columns: {missing}')
    usecols = [names[column] for column in required]
    matches = []
    for chunk in pd.read_csv(data_dir / filename, usecols=usecols, chunksize=chunksize):
        chunk.columns = [str(column).strip().lower() for column in chunk.columns]
        chunk = parse_nullable_numeric(chunk, ['subject_id', 'hadm_id', 'icustay_id', 'itemid', 'valuenum'])
        chunk = parse_timestamps(chunk, ['charttime'])
        chunk = chunk[chunk['itemid'].isin(item_ids) & chunk['valuenum'].notna()]
        if not chunk.empty:
            matches.append(chunk)
    return pd.concat(matches, ignore_index=True) if matches else pd.DataFrame(columns=required)


## Features and provisional label
The retained item IDs describe the original feature and procedure selection. A procedure-derived `label_vent` is a provisional retrospective ICU-stay label, not a validated future prediction target. Temporal label timing, censoring, and patient-level splitting require a complete study implementation.


In [ ]:
def add_provisional_label(icustays, procedureevents):
    result = icustays.copy()
    vent_stays = set(procedureevents.loc[procedureevents['itemid'].eq(225792), 'icustay_id'].dropna())
    result['label_vent'] = result['icustay_id'].isin(vent_stays).astype('int8')
    return result

def build_lstm_model(window_size, n_features, dropout_rate=0.3):
    import tensorflow as tf
    from tensorflow.keras import Sequential
    from tensorflow.keras.layers import Input, LSTM, Dropout, Dense
    from tensorflow.keras.optimizers import Adam
    model = Sequential([Input(shape=(window_size, n_features)), LSTM(64, return_sequences=True), Dropout(dropout_rate), LSTM(32), Dropout(dropout_rate), Dense(16, activation='relu'), Dropout(dropout_rate), Dense(1, activation='sigmoid')])
    model.compile(optimizer=Adam(learning_rate=1e-3), loss='binary_crossentropy', metrics=[tf.keras.metrics.AUC(name='auroc'), tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall'), 'accuracy'])
    return model

def mc_dropout_predict(model, X, n_samples=N_MC_SAMPLES):
    return np.asarray([model(X, training=True).numpy().ravel() for _ in range(n_samples)])


## Status
Model and MC Dropout functions are definitions only. No model is instantiated, no training or evaluation runs, and no outputs or metrics are fabricated.
